In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

In [ ]:
df = pd.read_csv("dirty_financial_transactions.csv")

In [ ]:
print(f'dataset shape: {df.shape}')

In [ ]:
display(df.sample(10, random_state=42))

In [ ]:
df.info()

In [ ]:
audit = pd.DataFrame({
    "dtype": df.dtypes,
    "missing": df.isna().sum(),
    "missing_%": df.isna().mean().mul(100).round(2),
    "unique": df.nunique()
}).sort_values("missing_%", ascending=False)

display(audit)

In [ ]:
initial_shape = df.shape
initial_missing = df.isna().sum().sum()

In [ ]:
df.columns=(
    df.columns
    .str.lower()
    .str.strip()
)

print(df.columns.to_list())

In [ ]:
df = df.drop(columns=['transaction_id'])

df['customer_id']=(
    df['customer_id']
    .str.extract(r'(\d+)')
    .astype('Int64'))

In [ ]:
df['transaction_date']=df['transaction_date'].replace('2023-13-01','2023-01-13')


leap_year_mask=(
    df['transaction_date']
    .str.contains(
        '02-29|02-30',
        na=False
    )
)

df.loc[leap_year_mask,'transaction_date']='2024-02-29'

df['transaction_date']=pd.to_datetime(
    df['transaction_date'],
    errors='coerce'
)

In [ ]:
df['product_name'].unique()

In [ ]:
df['product_name']=(
    df['product_name']
    .str.lower()
    .str.strip()
    )

In [ ]:
product_map = {
    'headphones': 'headphones',
    'coffee': 'coffee machine',
    'tablet': 'tablet',
    'tab': 'tablet',
    'coffee machine': 'coffee machine',
    'smartphone': 'smartphone',
    'laptop': 'laptop',
    'coffee ma': 'coffee machine',
    'cof': 'coffee machine',
    'smar': 'smartphone',
    'coffee m': 'coffee machine',
    't': 'tablet',
    'smartp': 'smartphone',
    'headp': 'headphones',
    'smart': 'smartphone',
    'la': 'laptop',
    'lapt': 'laptop',
    'tabl': 'tablet',
    'l': 'laptop',
    'c': 'coffee machine',
    'smartph': 'smartphone',
    'hea': 'headphones',
    'head': 'headphones',
    'smartpho': 'smartphone',
    'lapto': 'laptop',
    'headphon': 'headphones',
    'table': 'tablet',
    'co': 'coffee machine',
    'headphone': 'headphones',
    'coffee mac': 'coffee machine',
    'sm': 'smartphone',
    'headph': 'headphones',
    's': 'smartphone',
    'coffee mach': 'coffee machine',
    'smartphon': 'smartphone',
    'headpho': 'headphones',
    'coffee machin': 'coffee machine',
    'coff': 'coffee machine',
    'lap': 'laptop',
    'h': 'headphones',
    'he': 'headphones',
    'ta': 'tablet',
    'coffee machi': 'coffee machine',
    'coffe': 'coffee machine',
    'sma': 'smartphone'
}

In [ ]:
df['product_name']=(
    df['product_name']
    .replace(product_map)
    .astype('category')
    )

print("Remaining unique product values:")
df['product_name'].value_counts()

In [ ]:
assert df["product_name"].isin([
    "tablet",
    "smartphone",
    "laptop",
    "headphones",
    "coffee machine"
]).all()

In [ ]:
df['quantity'] = (
    df['quantity']
    .abs()
    .astype('Int64')
)

In [ ]:
df['price']=(
    df['price']
    .str.replace(r'[^0-9.\-]', '', regex=True)  # regex to remove non-numeric characters
    .str.replace(r'^\.', '0.', regex=True)      # handle ".99" → "0.99" 
    .astype('float32')
    .abs()
    .round(2)
)

In [ ]:
df['price'] = df.groupby('product_name',observed=True)['price'].transform(
    lambda x: x.fillna(x.mean().round(2))
)

In [ ]:
df['payment_method'].unique()

In [ ]:
df['transaction_status'].unique()

In [ ]:
df['payment_method']=(
    df['payment_method']
    .str.lower()
    .str.strip()
    .str.replace(' ','_')
    .replace({'pay_pal':'paypal','creditcard':'credit_card'})
    .astype('category')
)

df['payment_method'].value_counts()

In [ ]:
df['transaction_status']=(
    df['transaction_status']
    .str.lower()
    .str.strip()
    .replace('complete','completed')
    .astype('category')
)

df['transaction_status'].value_counts()

In [ ]:
rng=np.random.default_rng(1984)

In [ ]:
pre_fill_status_dist=df.groupby('payment_method',observed=True)['transaction_status'].value_counts(normalize=True)

display(pre_fill_status_dist)

In [ ]:
for method, group in df.groupby("payment_method",observed=True):
    mask = (
        (df["payment_method"] == method)
        & df["transaction_status"].isna()
    )

    distribution = (
        group["transaction_status"]
        .dropna()
        .value_counts(normalize=True)
    )

    df.loc[mask, "transaction_status"] = rng.choice(
        distribution.index,
        size=mask.sum(),
        p=distribution.values
    )

In [ ]:
df["transaction_status"].isna().sum()

In [ ]:
df['transaction_status'].value_counts(dropna=False)

In [ ]:
display(pre_fill_status_dist)

In [ ]:
post_fill_status_dist=df.groupby('payment_method',observed=True)['transaction_status'].value_counts(normalize=True)

display(post_fill_status_dist)

In [ ]:
print("Shape:", df.shape)

display(
    pd.DataFrame({
        "dtype": df.dtypes,
        "missing": df.isna().sum(),
        "missing_%": df.isna().mean().mul(100).round(2),
        "unique": df.nunique()
    })
)

In [ ]:
assert df["product_name"].notna().all()
assert df["price"].notna().all()
assert df["payment_method"].notna().all()
assert df["transaction_status"].notna().all()

assert (df["quantity"].dropna() >= 0).all()
assert (df["price"].dropna() >= 0).all()

assert pd.api.types.is_datetime64_any_dtype(df["transaction_date"])

In [ ]:
final_shape = df.shape
final_missing = df.isna().sum().sum()

print(f"Initial rows: {initial_shape[0]}")
print(f"Final rows:   {final_shape[0]}")
print(f"Rows removed: {initial_shape[0] - final_shape[0]}")

print(f"\nInitial missing values: {initial_missing}")
print(f"Final missing values:   {final_missing}")